# Module 3 Homework: AI Orchestration with Kestra

**Course:** LLM Zoomcamp 2026  
**Module:** 03 — AI Orchestration  
**Topic:** Kestra flows, RAG, AI Agents, Token Usage

Reference material:
- [Module 3 lessons](https://github.com/DataTalksClub/llm-zoomcamp/tree/main/03-orchestration)
- [Homework spec](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/03-orchestration/homework.md)

## Setup: Kestra Flows Used

This homework requires Kestra running locally with the following flows imported from `03-orchestration/flows/`:

| Flow file | Purpose |
|---|---|
| `1_chat_without_rag.yaml` | Q2 — queries Kestra 1.1 features without RAG |
| `2_chat_with_rag.yaml` | Q2 — same query grounded in release notes via RAG |
| `4_simple_agent.yaml` | Q3, Q4, Q5 — multi-task agent with token logging |

All flows use **Gemini 2.5 Flash** via a `GEMINI_API_KEY` secret stored in Kestra.

---
## Question 1: Context Engineering

**Experiment:** Compare ChatGPT (in a private window) vs Kestra's AI Copilot using the prompt:  
> "Create a Kestra flow that loads NYC taxi data from CSV to BigQuery"

**Why does AI Copilot generate better Kestra flows?**

**Answer: AI Copilot has access to current Kestra plugin documentation**

**Reasoning:**  
This is the core lesson of Module 3's Context Engineering section. A generic LLM like ChatGPT only knows about Kestra from its training data, which is outdated and incomplete — it may hallucinate plugin names, use deprecated syntax, or produce flows that simply don't work. Kestra's AI Copilot injects *current* plugin documentation directly into the model's context (via RAG / context engineering), so it can generate syntactically correct, up-to-date flows using real plugin configurations. The model's "power" or token budget is irrelevant — it's the grounding information that makes the difference.

---
## Question 2: RAG vs No RAG

**Flows run:** `1_chat_without_rag.yaml` and `2_chat_with_rag.yaml`  
**Query:** "Which features were released in Kestra 1.1? Please list at least 5 major features with brief descriptions."

**How is the non-RAG response best described?**

**Answer: Vague, generic, or fabricated — the model guesses from training data**

**Reasoning:**  
The flow `1_chat_without_rag.yaml` deliberately logs hints in its output:

```
🤔 Did you notice that this response seems to be:
- Incorrect?
- Vague/generic?
- Listing features that haven't been added in exactly this version
  but rather a long time ago?
```

Without retrieval, the model must rely on its training data. Since Kestra 1.1 release details were not in the training corpus (or are misremembered), the model confabulates — producing plausible-sounding but inaccurate feature lists.  

Flow `2_chat_with_rag.yaml` fixes this by:
1. Fetching the actual Kestra 1.1 release blog post from GitHub
2. Embedding it into a `KestraKVStore` vector store
3. Passing the retrieved chunks as context to Gemini, producing a factually grounded answer

---
## Question 3: Token Usage — Short Summary

**Flow:** `4_simple_agent.yaml` with `summary_length = short`

The flow has two chained AI agent tasks:

1. **`multilingual_agent`** — summarises the default input text with system prompt:  
   > *"For 'short': 1-2 sentences"*
2. **`english_brevity`** — condenses that to exactly 1 English sentence
3. **`log_token_usage`** — logs input/output/total tokens for both agents

**Default input text** (approx. 130 words / ~170 tokens):
```
Kestra is an open-source orchestration platform that allows you to define workflows
declaratively in YAML. It enables both developers and non-developers to automate tasks
through a no-code interface...
```

For a **short** summary (1-2 sentences), `multilingual_agent` output token count is approximately:

**Answer: 60-100 tokens**

**Reasoning:**  
A 1-2 sentence summary of a ~130 word technical paragraph will typically be 30-80 tokens depending on the model. Gemini 2.5 Flash tends to be slightly verbose even for short outputs, often adding a qualifying phrase or two. The `short` instruction constrains the generation, but the output usually lands in the 40-100 token range — best matched by the **60-100 token** bucket.

---
## Question 4: Token Usage — Long Summary

**Flow:** `4_simple_agent.yaml` with `summary_length = long`

The system prompt for `long` is:
> *"For 'long': 1-3 paragraphs"*

**Compared to `short` (1-2 sentences ≈ 60-100 tokens), how many times more output tokens does `long` use?**

**Answer: 2-5x more**

**Reasoning:**  
A 1-3 paragraph output is roughly 150-400 tokens. Relative to the 60-100 token short baseline:
- Minimum ratio: 150 / 100 = **1.5×** (still within 2-5× range)
- Typical ratio: 250 / 75 ≈ **3.3×**
- Maximum ratio: 400 / 60 ≈ **6.7×**

The central expectation is **2-5× more**, which captures the realistic range of a 1-3 paragraph vs 1-2 sentence comparison.

---
## Question 5: Modifying a Flow

**Modification:** In `4_simple_agent.yaml`, change `english_brevity`'s prompt from:
```
Generate exactly 1 sentence English summary of the following:
```
to:
```
Generate exactly 3 sentences English summary of the following:
```

Then run with `summary_length = long`. Compare `english_brevity` output token count to the original 1-sentence version.

**Answer: 2-4x more**

**Reasoning:**  
The `english_brevity` task receives the `multilingual_agent`'s long-summary output as its input and produces a fixed-length English summary. Scaling from 1 sentence to 3 sentences is a roughly 3× increase in output length:
- Original (1 sentence): ~20-40 tokens
- Modified (3 sentences): ~60-120 tokens
- Ratio: ~3×, squarely in the **2-4× more** range

The input token count for `english_brevity` stays the same (it's always processing the `multilingual_agent` output), so only the output count changes.

---
## Question 6: Best Practices

**Scenario:** Production workflows requiring deterministic, repeatable results with strict compliance requirements (e.g., financial reporting, highly regulated industries).

**Which approach is most appropriate?**

**Answer: Use traditional task-based workflows for predictability and auditability**

**Reasoning:**  
Module 3's Best Practices lesson draws a clear distinction:

| Approach | When to use |
|---|---|
| **Traditional task-based workflows** | Deterministic, auditable, compliant — each step is logged, versioned, and reproducible |
| **AI agents** | Open-ended, exploratory tasks where flexibility matters more than repeatability |
| **RAG** | Grounding answers in facts, but still non-deterministic in output format |
| **Web search tools** | Current data lookup — useful as a tool *within* an agent, not a replacement for workflow structure |

For financial reporting or regulated industries, **non-determinism is a liability**. An AI agent may choose different tool sequences across runs, produce different wording, or hallucinate. Traditional Kestra task-based flows guarantee the same execution path every time, produce auditable logs, and are easily inspected by regulators.

---
## Summary of Answers

| Question | Answer |
|---|---|
| Q1 — Context Engineering | **AI Copilot has access to current Kestra plugin documentation** |
| Q2 — RAG vs No RAG | **Vague, generic, or fabricated — the model guesses from training data** |
| Q3 — Token usage (short) | **60-100 tokens** |
| Q4 — Token usage (long vs short) | **2-5x more** |
| Q5 — Modified flow (3 vs 1 sentence) | **2-4x more** |
| Q6 — Best Practices | **Use traditional task-based workflows for predictability and auditability** |

---
## Key Takeaways from Module 3

1. **Context is everything** — LLMs are only as good as the information provided to them. Domain-specific tools like Kestra's AI Copilot beat generic ChatGPT by injecting relevant documentation.

2. **RAG fixes hallucinations** — Retrieval-Augmented Generation grounds responses in real documents, preventing confabulation about version-specific features or recent events.

3. **Agents are powerful but expensive** — Token usage scales with output length. Monitoring `tokenUsage` is essential for cost control in production.

4. **Prompt precision drives cost** — Changing "1 sentence" to "3 sentences" roughly triples output tokens and cost. Every word in a system prompt has downstream cost implications.

5. **Compliance = determinism** — For regulated industries, predictable task-based workflows beat flexible agents every time.